# 🎓 Yonsei Colab Studio — 교육용 소형 LLM 파인튜닝

Google Colab **무료 T4 GPU**에서 [Unsloth](https://github.com/unslothai/unsloth)로 소형 LLM을 파인튜닝합니다.
메모리 사용량 ~80% 절감, 속도 2~3배로 무료 코랩에서도 OOM 없이 학습됩니다.

- **목적**: 20가지 교육 시나리오(소크라테스 문답 · 단계별 채점 · 학부모 안내문 변환 등) 특화
- **기본 모델**: `unsloth/Qwen2.5-1.5B-Instruct` (한국어 우수, 초경량)
- **방식**: LoRA(QLoRA) 4bit

> ⚠️ 먼저 상단 메뉴에서 **런타임 → 런타임 유형 변경 → T4 GPU** 를 선택하세요.


## 0. GPU 확인


In [2]:
!nvidia-smi


Sat Jun  6 12:17:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Unsloth 설치
코랩 최신 환경에 맞춰 자동 설치됩니다. (1~2분 소요)


In [3]:
%%capture
import os
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets


## 2. 학습 데이터 — HuggingFace Hub에서 받기
손으로 만든 예시가 아니라 **실제 한국어 교육 데이터셋**을 HF Hub에서 받아 20시나리오 포맷으로 변환합니다.

추천 데이터셋(소스별 N개만 streaming 추출 → 대형도 전체 다운로드 없음):
- `JosephLee/korean-socratic-qa` (105k) — 소크라테스 문답
- `kuotient/orca-math-word-problems-193k-korean` (193k) — 수학 단계 풀이
- `beomi/KoAlpaca-v1.1a` (21k) — 일반 instruction 베이스
- `jojo0217/korean_safe_conversation` (27k, Apache-2.0) — 정서지원·공감
- `neuralfoundry-coder/aihub-korean-education-instruct-sample` (6k) — 교육 상담·분석


In [4]:
# 저장소 clone (스크립트/레지스트리 사용)
REPO = 'https://github.com/xide-projext/xideproject.git'
import os
if not os.path.exists('xideproject'):
    !git clone -q $REPO
%cd xideproject

# HF에서 소스별 8000개씩 받아 data/hf_train.jsonl 생성 (≈5분)
!python scripts/fetch_hf_datasets.py --per-source 8000 --val-ratio 0.05

from pathlib import Path
DATA_PATH = "data/hf_train.jsonl"
assert Path(DATA_PATH).exists(), "fetch 실패 — 위 셀 로그 확인"
print("학습 데이터 준비 완료:", DATA_PATH)


/content/xideproject

▶ [general] beomi/KoAlpaca-v1.1a (목표 8000개, license=CC-BY-NC-4.0(추정))
README.md: 100% 1.75k/1.75k [00:00<00:00, 1.96MB/s]
  ✅ 8000개 수집

▶ [socratic] JosephLee/korean-socratic-qa (목표 8000개, license=확인필요)
README.md: 100% 1.24k/1.24k [00:00<00:00, 2.38MB/s]
Repo card metadata block was not found. Setting CardData to empty.
  ✅ 8000개 수집

▶ [math] kuotient/orca-math-word-problems-193k-korean (목표 8000개, license=CC-BY-SA-4.0)
README.md: 100% 1.13k/1.13k [00:00<00:00, 2.59MB/s]
  ✅ 8000개 수집

▶ [empathy] jojo0217/korean_safe_conversation (목표 8000개, license=Apache-2.0)
README.md: 100% 1.93k/1.93k [00:00<00:00, 2.29MB/s]
  ✅ 8000개 수집

▶ [edu] neuralfoundry-coder/aihub-korean-education-instruct-sample (목표 8000개, license=CC-BY-NC-SA-4.0)
README.md: 100% 2.00k/2.00k [00:00<00:00, 5.56MB/s]
  ✅ 5400개 수집

▶ [roleplay] huggingface-KREW/korean-role-playing (목표 8000개, license=Apache-2.0)
README.md: 100% 5.88k/5.88k [00:00<00:00, 12.6MB/s]
  ✅ 8000개 수집

▶ [translation] bawin/korean-e

## 3. 모델 로드 (4bit)


In [5]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct"  # 더 작게: Qwen2.5-0.5B-Instruct / 다른 계열: Llama-3.2-1B-Instruct

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,            # 자동 감지 (T4=float16)
    load_in_4bit = True,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.1: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


## 4. LoRA 어댑터 추가
전체 가중치가 아닌 일부 행렬만 학습해 메모리/시간을 절약합니다.


In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)


Unsloth 2026.6.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


## 5. 데이터 포맷팅 (채팅 템플릿)
`instruction`(시나리오 지시) → system, `input` → user, `output` → assistant 로 매핑합니다.


In [7]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def to_text(ex):
    msgs = [
        {"role": "system",    "content": ex["instruction"]},
        {"role": "user",      "content": ex["input"]},
        {"role": "assistant", "content": ex["output"]},
    ]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}

dataset = load_dataset("json", data_files=DATA_PATH, split="train")
dataset = dataset.map(to_text)
print("샘플 수:", len(dataset))
print(dataset[0]["text"][:600])


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/86805 [00:00<?, ? examples/s]

샘플 수: 86805
<|im_start|>system
시나리오[5]: 주어진 지문을 근거로 질문에 정확히 답하세요. 지문에 없는 내용은 지어내지 않습니다.<|im_end|>
<|im_start|>user
[지문]
한편 북원의 성주 양길은 후백제와 동맹을 맺고, 후백제의 견훤은 양길을 대장군으로 임명하여 후고구려를 협공한다. 왕건은 군사 일부를 북원성으로 보내 양길의 군사와 대적하게 하는 한편 직접 수군을 이끌고 후백제의 목포, 신안, 나주 일대를 공격한다. 그 과정에서 나주의 유력 호족이며 오부돈의 아들인 오다련군 일파를 포섭한다. 후백제의 민심이 이반된 틈을 타 왕건은 서남해안을 공략하였고 오다련군 등 서남의 귀족들은 왕건에게 투항하였다. 갑판 선상에서 시내 위를 바라보던 왕건이 오색(五色)의 운기를 보고 달려갔다가 빨래하고 있는 오씨를 보았다.

그가 나주에 이르러 한 우물가에 있던 처녀에게 물을 달라 하였는데, 처녀는 바가지에 물을 떠주면서 버들잎을 하나 띄워서 주었다. 이유를 묻자 그가 빨리 마시고 체할 것이 염려된다는 것이었다. 그는 나주의 호족 오다련군을 만나게 되었고, 오다련군은 그의 사람됨됨이가 범상치 않음을 알아보고 자신의 딸 오씨를 그와 결혼시킨다. 그 둘 사


## 6. 트레이너 설정 & 학습
데모용 `max_steps=60`. 실제 학습에선 `num_train_epochs`로 바꾸세요.


In [13]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LENGTH,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,                # 또는 num_train_epochs = 3 (데이터 많을 때)
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)
trainer_stats = trainer.train()


Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/86805 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 86,805 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.151107
2,1.870544
3,1.736588
4,1.950920
5,1.640570
6,1.145037
7,1.311152
8,1.911540
9,1.662424
10,1.668103


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


## 7. 추론 테스트
학습한 시나리오 톤이 나오는지 즉석에서 확인합니다.


In [14]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "시나리오[1]: 소크라테스식 문답법으로 학생을 가이드하세요. 정답을 바로 주지 말고 힌트와 역질문으로 스스로 답을 찾게 유도합니다."},
    {"role": "user",   "content": "선생님, 삼각형 내각의 합이 왜 180도예요?"},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

out = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.7, do_sample=True)
print(tokenizer.decode(out[0], skip_special_tokens=True))


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


system
시나리오[1]: 소크라테스식 문답법으로 학생을 가이드하세요. 정답을 바로 주지 말고 힌트와 역질문으로 스스로 답을 찾게 유도합니다.
user
선생님, 삼각형 내각의 합이 왜 180도예요?
assistant
(더 분명히 말하기) 삼각형에 있는 각도가 왜 180도로 총합이 되는지 이해하려면?


## 8. 저장 & 내보내기
- **LoRA 어댑터**: 가볍게 재사용 (수십 MB)
- **GGUF**: Ollama / LM Studio / llama.cpp 용 (옵션, 시간 소요)


In [15]:
# (1) LoRA 어댑터만 저장
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
print("LoRA 저장 완료 → lora_model/")

# (2) GGUF 내보내기 (필요할 때 주석 해제)
# model.save_pretrained_gguf("model_gguf", tokenizer, quantization_method="q4_k_m")

# (3) 구글 드라이브에 백업 (필요할 때 주석 해제)
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r lora_model /content/drive/MyDrive/


LoRA 저장 완료 → lora_model/


---
### 다음 단계
1. 데이터를 더 받으려면 `--per-source` 를 키우세요 (예: 20000). 소스 추가는 `scripts/fetch_hf_datasets.py` 의 SOURCES 레지스트리에서.
2. 특정 시나리오만 학습하려면 `--only socratic math` 처럼 지정.
3. 시나리오별 성능은 `data/scenarios.json` 의 *기대치(expectation)* 기준으로 평가하세요.
4. `data/seed_train.jsonl` 은 학습용이 아니라 시나리오별 **목표 톤 예시(참고용)** 입니다.


In [16]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048

# 순수한 기본 모델 로드
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(base_model)

messages = [
    {
        "role": "system",
        "content": "시나리오[8]: 코딩 디버깅 가이드 (힌트 제공형). 정답 소스코드를 제공을 하지 않고, 힌트만 제공하세요"
    },
    {
            "role": "user",
            "content": (
                "import numpy as np\n"
                "import matplotlib.pyplot as plt\n\n"
                "x = np.linspace(1, 6, 10)\n\n"
                "plt.figure()\n"
                "for i in range(10):\n"
                "    T = i + 20\n"
                "    Fit = 4 * x - ( 2 + i)\n"
                "    noise = np.random.normal(loc=0, scale=1.0, size=len(Fit))\n"
                "    data_exp = Fit + noise/5\n"
                "    plt.scatter(x, data_exp, marker='o', label= f'Experimental data T = {T}°C')\n"
                "    plt.plot(x, Fit, 'k-', label=f'Fit T = {T}°C')\n"
                "plt.xlabel('Optical power (mW)')\n"
                "plt.ylabel('Current (mA)')\n"
                "plt.legend()\n"
                "plt.grid(True)\n"
                "plt.show()\n\n"
                "I wrote an example of my problem in the code above and I uploaded the graph it produces. "
                "What I want to do is to reduce the legend size because I can have many temperature, and if the legend "
                "is too big it can hide the data and the fit. So to do this I want to display a single label for the fit "
                "because they appear the same way (black lines). It's useless to write '--- Fit 20°C', '--- FIT 21°C', "
                "'---FIT 22°'..etc Instead I just want to plot '--- FIT' once.\n\n answer this in korean"
            )
        }
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = base_model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.7)

print("=" * 50)
print("[진짜 BEFORE] 파인튜닝 전혀 안 된 기본 모델 답변:")
print("=" * 50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant")[-1].strip())

==((====))==  Unsloth 2026.6.1: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[진짜 BEFORE] 파인튜닝 전혀 안 된 기본 모델 답변:
그림이 생성되는 과정과 결과에 대해 설명해주시면 좋습니다. 그림의 제목, X축, Y축의 이름 등을 알려주실 수 있나요? 그리고 그래프가 어떤 데이터로 구성되어 있는지, 각 데이터의 특성 등도 알려주시면 더 도움이 될 것 같습니다. 추가적으로 필요한 정보가 있다면 알려주세요.


In [17]:
# 내가 학습시켜 저장한 lora_model 폴더 지정
tuned_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "lora_model",
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(tuned_model)

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = tuned_model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.7)

print("=" * 50)
print("[진짜 AFTER] 내가 학습시킨 모델 답변:")
print("=" * 50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant")[-1].strip())

==((====))==  Unsloth 2026.6.1: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[진짜 AFTER] 내가 학습시킨 모델 답변:
여기와 같은 그래프의 레이블 크기를 줄이는 방법을 알려드립니다. 그래프에 표시할 레이블의 크기를 줄이는 것은 매우 유용한 작업입니다. 그래프에서 여러 종류의 데이터를 비교하거나 분석하는 데 사용되는 레이블은 항상 큰 것이 좋습니다. 그러나 그게 어렵다면, 특정 레이블이 다른 레이블과 동일한 형태로 표현되어서는 안 됩니다. 이 문제에 대한 예시 코드가 있습니다:
